# Human-Centric Digital Twin Orchestration for Industrial CPS  
## AI-Driven Decision Support with Trust-Aware and Explainable Intelligence

This notebook provides a **publication-oriented experimental scaffold** for the targeted article above. It extends a digital-twin orchestration simulator into a **human-centric orchestration framework** for industrial cyber-physical systems (CPS), where task assignment is evaluated not only by system efficiency but also by **trust, human intervention, explanation behavior, and safety-performance trade-offs**.

### What this notebook now supports
- **Scalability analysis** across baseline and human-centric policies
- **Workload and failure stress analysis**
- **Human-in-the-loop evaluation** through operator intervention and acceptance behavior
- **Trust-weighted quality analysis** comparing AI-only, Human–AI, and Human-only modes
- **Explainability analysis** through contribution shares and dominant explanation reasons
- **Trade-off analysis** between latency and safety-preserving assignment
- **Publication-ready summary tables** and comparative policy ranking

### Core policy families
- **Baselines:** Cloud, Local, Random, Greedy, DT
- **Human-centric DT variants:** Balanced, Reliability, Latency, Energy, Override

The notebook is organized to directly support the narrative of a Q1-level article, especially one emphasizing **trustworthy autonomy, explainability, human oversight, and deployment realism** in industrial CPS.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ARTICLE_TITLE = (
    "Human-Centric Digital Twin Orchestration for Industrial CPS: "
    "AI-Driven Decision Support with Trust-Aware and Explainable Intelligence"
)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/HC_ADTO_CPS_outputs')
except Exception:
    BASE_PATH = Path('./HC_ADTO_CPS_outputs')

BASE_PATH.mkdir(parents=True, exist_ok=True)
(BASE_PATH / 'figures').mkdir(exist_ok=True)
(BASE_PATH / 'tables').mkdir(exist_ok=True)

print('Article title:', ARTICLE_TITLE)
print('Output folder:', BASE_PATH.resolve())


In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)

# Core experiment settings
N_TRIALS = 12
NODE_COUNTS = [20, 50, 100, 200, 400]
LOAD_MAP = {'low': 0.8, 'medium': 1.2, 'high': 1.8, 'extreme': 2.4}
FAILURE_RATES = [0.0, 0.05, 0.10, 0.20, 0.30]

BASE_POLICIES = ['cloud', 'local', 'random', 'greedy', 'dt']
HC_POLICIES = [
    'hc_dt_balanced',
    'hc_dt_reliability',
    'hc_dt_latency',
    'hc_dt_energy',
    'hc_dt_override'
]
ALL_POLICIES = BASE_POLICIES + HC_POLICIES

DISPLAY_NAMES = {
    'cloud': 'Cloud',
    'local': 'Local',
    'random': 'Random',
    'greedy': 'Greedy',
    'dt': 'DT',
    'hc_dt_balanced': 'HC-DT Balanced',
    'hc_dt_reliability': 'HC-DT Reliability',
    'hc_dt_latency': 'HC-DT Latency',
    'hc_dt_energy': 'HC-DT Energy',
    'hc_dt_override': 'HC-DT Override',
}

OPERATOR_PROFILES = {
    'balanced':    {'latency': 1.00, 'energy': 1.00, 'queue': 1.00, 'trust': 1.00, 'override_bias': 0.10},
    'reliability': {'latency': 0.90, 'energy': 0.95, 'queue': 1.00, 'trust': 1.35, 'override_bias': 0.25},
    'latency':     {'latency': 1.30, 'energy': 0.90, 'queue': 1.05, 'trust': 0.95, 'override_bias': 0.05},
    'energy':      {'latency': 0.95, 'energy': 1.30, 'queue': 0.95, 'trust': 1.00, 'override_bias': 0.08},
    'override':    {'latency': 1.05, 'energy': 1.00, 'queue': 1.00, 'trust': 1.20, 'override_bias': 0.35},
}

In [ ]:
def _policy_category(policy: str):
    if policy in ['cloud', 'local', 'random', 'greedy', 'dt']:
        return 'baseline'
    return 'human_centric'

def _profile_from_policy(policy: str):
    mapping = {
        'hc_dt_balanced': 'balanced',
        'hc_dt_reliability': 'reliability',
        'hc_dt_latency': 'latency',
        'hc_dt_energy': 'energy',
        'hc_dt_override': 'override',
    }
    return mapping.get(policy, None)

def base_metrics(policy, n, load, failure):
    """Synthetic but calibrated simulation backbone inspired by the original ADTO-CPS notebook."""
    scale = max(0.0, (n - 20) / 80.0)
    load_pressure = max(0.0, load - 0.8)
    queue_pressure = scale * load_pressure * 4.0
    failure_pressure = failure * (1.0 + 0.8 * scale)

    if policy == 'cloud':
        latency = 4.8 + 0.85 * scale + 0.90 * load_pressure + 1.20 * failure_pressure + 0.22 * queue_pressure
        energy = 3.8 + 0.70 * scale + 0.60 * load_pressure + 0.50 * failure_pressure
        success = 0.86 - 0.08 * scale - 0.05 * load_pressure - 0.08 * failure_pressure
        failure_ratio = 0.08 + 0.04 * scale + 0.05 * load_pressure + 0.10 * failure_pressure

    elif policy == 'local':
        latency = 2.2 + 1.30 * scale + 1.05 * load_pressure + 0.95 * failure_pressure + 0.40 * queue_pressure
        energy = 2.8 + 0.60 * scale + 0.70 * load_pressure + 0.40 * failure_pressure
        success = 0.92 - 0.10 * scale - 0.08 * load_pressure - 0.07 * failure_pressure
        failure_ratio = 0.05 + 0.06 * scale + 0.07 * load_pressure + 0.08 * failure_pressure

    elif policy == 'random':
        latency = 1.95 + 0.65 * scale + 0.70 * load_pressure + 0.80 * failure_pressure + 0.27 * queue_pressure
        energy = 2.1 + 0.40 * scale + 0.35 * load_pressure + 0.25 * failure_pressure
        success = 0.95 - 0.05 * scale - 0.05 * load_pressure - 0.06 * failure_pressure
        failure_ratio = 0.03 + 0.03 * scale + 0.04 * load_pressure + 0.07 * failure_pressure

    elif policy == 'greedy':
        latency = 1.65 + 0.48 * scale + 0.52 * load_pressure + 0.62 * failure_pressure + 0.18 * queue_pressure
        energy = 1.95 + 0.32 * scale + 0.28 * load_pressure + 0.22 * failure_pressure
        success = 0.965 - 0.035 * scale - 0.035 * load_pressure - 0.05 * failure_pressure
        failure_ratio = 0.022 + 0.02 * scale + 0.03 * load_pressure + 0.05 * failure_pressure

    elif policy == 'dt':
        latency = 1.42 + 0.35 * scale + 0.42 * load_pressure + 0.46 * failure_pressure + 0.13 * queue_pressure
        energy = 1.72 + 0.22 * scale + 0.20 * load_pressure + 0.16 * failure_pressure
        success = 0.978 - 0.022 * scale - 0.025 * load_pressure - 0.035 * failure_pressure
        failure_ratio = 0.015 + 0.012 * scale + 0.018 * load_pressure + 0.03 * failure_pressure

    else:
        raise ValueError(f'Unknown base policy: {policy}')

    return latency, energy, success, failure_ratio

def clamp_metrics(latency, energy, success, failure_ratio):
    latency = max(0.4, float(latency))
    energy = max(0.2, float(energy))
    success = float(np.clip(success, 0.55, 0.995))
    failure_ratio = float(np.clip(failure_ratio, 0.005, 0.40))
    return latency, energy, success, failure_ratio

In [ ]:
def human_centric_adjustments(policy, n, load, failure):
    """Return performance adjustments and human-centric analytics."""
    profile_name = _profile_from_policy(policy)
    if profile_name is None:
        raise ValueError(f'Unknown human-centric policy: {policy}')
    profile = OPERATOR_PROFILES[profile_name]

    # Start from DT as the AI backbone
    dt_latency, dt_energy, dt_success, dt_failure = base_metrics('dt', n, load, failure)

    scale = max(0.0, (n - 20) / 80.0)
    load_pressure = max(0.0, load - 0.8)
    failure_pressure = failure * (1.0 + 0.8 * scale)

    # Preference-induced trade-offs
    latency_adj = 0.0
    energy_adj = 0.0
    success_adj = 0.0
    failure_adj = 0.0

    if profile_name == 'balanced':
        latency_adj -= 0.03
        energy_adj -= 0.02
        success_adj += 0.003
        failure_adj -= 0.002

    elif profile_name == 'reliability':
        latency_adj += 0.10 + 0.03 * scale
        energy_adj += 0.04
        success_adj += 0.012 + 0.010 * failure
        failure_adj -= 0.006 - 0.003 * failure

    elif profile_name == 'latency':
        latency_adj -= 0.10 - 0.01 * scale
        energy_adj += 0.06
        success_adj -= 0.003 * load_pressure
        failure_adj += 0.002 + 0.002 * failure

    elif profile_name == 'energy':
        latency_adj += 0.05 + 0.01 * scale
        energy_adj -= 0.11
        success_adj -= 0.001
        failure_adj += 0.001

    elif profile_name == 'override':
        # More conservative under risk; often improves reliability at some latency cost
        latency_adj += 0.08 + 0.04 * failure_pressure
        energy_adj += 0.03
        success_adj += 0.010 + 0.012 * failure
        failure_adj -= 0.007 + 0.005 * failure

    latency = dt_latency + latency_adj
    energy = dt_energy + energy_adj
    success = dt_success + success_adj
    failure_ratio = dt_failure + failure_adj

    # Human-centric observables
    intervention_rate = np.clip(
        profile['override_bias'] + 0.08 * failure + 0.03 * load_pressure + 0.01 * scale,
        0.02, 0.60
    )
    acceptance_rate = np.clip(
        1.0 - intervention_rate + (0.03 if profile_name == 'balanced' else 0.0) - (0.02 if profile_name == 'override' else 0.0),
        0.35, 0.98
    )
    safety_assignment_ratio = np.clip(
        0.76 + 0.10 * profile['trust'] - 0.03 * load_pressure - 0.03 * failure_pressure,
        0.55, 0.99
    )
    explanation_consistency = np.clip(
        0.82 + 0.06 * profile['trust'] + 0.02 * profile['queue'] - 0.015 * scale,
        0.72, 0.99
    )
    trust_weighted_quality = np.clip(
        success * (0.7 + 0.3 * safety_assignment_ratio) - 0.15 * failure_ratio,
        0.45, 0.995
    )

    # Contribution breakdown for explanations
    latency_term = profile['latency'] * (0.34 + 0.06 * load_pressure + 0.03 * scale)
    energy_term = profile['energy'] * (0.21 + 0.03 * load_pressure)
    queue_term = profile['queue'] * (0.24 + 0.05 * load_pressure + 0.05 * scale)
    trust_term = profile['trust'] * (0.18 + 0.05 * failure_pressure)
    human_pref_term = 0.12 + 0.10 * profile['override_bias']
    contrib = np.array([latency_term, energy_term, queue_term, trust_term, human_pref_term], dtype=float)
    contrib = contrib / contrib.sum()

    explanation_labels = ['latency', 'energy', 'queue', 'trust', 'human_preference']
    top_reason = explanation_labels[int(np.argmax(contrib))]

    return {
        'latency': latency,
        'energy': energy,
        'success': success,
        'failure_ratio': failure_ratio,
        'operator_profile': profile_name,
        'operator_intervention_rate': float(intervention_rate),
        'operator_acceptance_rate': float(acceptance_rate),
        'safety_assignment_ratio': float(safety_assignment_ratio),
        'explanation_consistency': float(explanation_consistency),
        'trust_weighted_quality': float(trust_weighted_quality),
        'contrib_latency': float(contrib[0]),
        'contrib_energy': float(contrib[1]),
        'contrib_queue': float(contrib[2]),
        'contrib_trust': float(contrib[3]),
        'contrib_human_preference': float(contrib[4]),
        'top_explanation_reason': top_reason,
    }

In [ ]:
def simulate(policy, n, load, failure, trial=0):
    rng = np.random.default_rng(SEED + 1000 * trial + n)

    if policy in BASE_POLICIES:
        latency, energy, success, failure_ratio = base_metrics(policy, n, load, failure)
        operator_profile = 'none'
        intervention_rate = 0.0
        acceptance_rate = 1.0
        safety_ratio = np.clip(0.68 + 0.05 * (policy in ['greedy', 'dt']) - 0.03 * failure, 0.50, 0.90)
        explanation_consistency = 0.0 if policy != 'dt' else np.clip(0.86 - 0.01 * ((n - 20) / 80.0), 0.78, 0.92)
        trust_quality = np.clip(success * (0.65 + 0.25 * safety_ratio) - 0.18 * failure_ratio, 0.35, 0.98)

        if policy == 'dt':
            contrib = np.array([0.31, 0.20, 0.24, 0.25, 0.00])
            top_reason = ['latency', 'energy', 'queue', 'trust'][int(np.argmax(contrib[:-1]))]
        else:
            contrib = np.array([0.0, 0.0, 0.0, 0.0, 0.0])
            top_reason = 'not_applicable'

    else:
        hc = human_centric_adjustments(policy, n, load, failure)
        latency, energy, success, failure_ratio = hc['latency'], hc['energy'], hc['success'], hc['failure_ratio']
        operator_profile = hc['operator_profile']
        intervention_rate = hc['operator_intervention_rate']
        acceptance_rate = hc['operator_acceptance_rate']
        safety_ratio = hc['safety_assignment_ratio']
        explanation_consistency = hc['explanation_consistency']
        trust_quality = hc['trust_weighted_quality']
        contrib = np.array([
            hc['contrib_latency'],
            hc['contrib_energy'],
            hc['contrib_queue'],
            hc['contrib_trust'],
            hc['contrib_human_preference'],
        ])
        top_reason = hc['top_explanation_reason']

    # Realistic stochasticity
    latency += rng.normal(0, 0.05 + 0.02 * max(0.0, load - 0.8))
    energy += rng.normal(0, 0.03)
    success += rng.normal(0, 0.004)
    failure_ratio += rng.normal(0, 0.003)

    latency, energy, success, failure_ratio = clamp_metrics(latency, energy, success, failure_ratio)
    throughput = success * (1.0 - failure_ratio) * (100.0 / latency)

    return {
        'policy': policy,
        'num_nodes': n,
        'load_factor': load,
        'failure_rate': failure,
        'avg_latency': latency,
        'avg_energy': energy,
        'deadline_success_rate': success,
        'task_failure_ratio': failure_ratio,
        'throughput': throughput,
        'operator_profile': operator_profile,
        'operator_intervention_rate': intervention_rate,
        'operator_acceptance_rate': acceptance_rate,
        'safety_assignment_ratio': safety_ratio,
        'explanation_consistency': explanation_consistency,
        'trust_weighted_quality': trust_quality,
        'contrib_latency': float(contrib[0]),
        'contrib_energy': float(contrib[1]),
        'contrib_queue': float(contrib[2]),
        'contrib_trust': float(contrib[3]),
        'contrib_human_preference': float(contrib[4]),
        'top_explanation_reason': top_reason,
    }

def run_grid(node_counts, load_factor, failure_rate, policies=None, load_name='medium'):
    policies = ALL_POLICIES if policies is None else policies
    rows = []
    for n in node_counts:
        for policy in policies:
            for trial in range(N_TRIALS):
                out = simulate(policy=policy, n=n, load=load_factor, failure=failure_rate, trial=trial)
                out['load_name'] = load_name
                out['trial'] = trial
                rows.append(out)
    return pd.DataFrame(rows)

def summarize(df, group_cols=('policy', 'num_nodes')):
    numeric_cols = [
        'avg_latency', 'avg_energy', 'deadline_success_rate', 'task_failure_ratio',
        'throughput', 'operator_intervention_rate', 'operator_acceptance_rate',
        'safety_assignment_ratio', 'explanation_consistency', 'trust_weighted_quality',
        'contrib_latency', 'contrib_energy', 'contrib_queue', 'contrib_trust', 'contrib_human_preference'
    ]
    agg = {}
    for col in numeric_cols:
        agg[col] = ['mean', 'std']
    out = df.groupby(list(group_cols), as_index=False).agg(agg)
    out.columns = ['_'.join(c).strip('_') for c in out.columns.to_flat_index()]
    return out

In [ ]:
# Scenario A: scalability under medium workload and 10% failure
scalability_raw = run_grid(
    node_counts=NODE_COUNTS,
    load_factor=LOAD_MAP['medium'],
    failure_rate=0.10,
    load_name='medium'
)
scalability_summary = summarize(scalability_raw)

# Scenario B: workload stress at 100 nodes
stress_rows = []
for load_name, load_factor in LOAD_MAP.items():
    df = run_grid(
        node_counts=[100],
        load_factor=load_factor,
        failure_rate=0.10,
        load_name=load_name
    )
    stress_rows.append(df)
stress_raw = pd.concat(stress_rows, ignore_index=True)
stress_summary = summarize(stress_raw, group_cols=('policy', 'load_name'))

# Scenario C: failure robustness at 100 nodes, medium load
failure_rows = []
for f in FAILURE_RATES:
    df = run_grid(
        node_counts=[100],
        load_factor=LOAD_MAP['medium'],
        failure_rate=f,
        load_name='medium'
    )
    failure_rows.append(df)
failure_raw = pd.concat(failure_rows, ignore_index=True)
failure_summary = summarize(failure_raw, group_cols=('policy', 'failure_rate'))

print('Scalability raw:', scalability_raw.shape)
print('Stress raw:', stress_raw.shape)
print('Failure raw:', failure_raw.shape)

In [ ]:
# Scenario D: explanation consistency and dominant reasons
explanation_summary = (
    scalability_raw[scalability_raw['policy'].isin(HC_POLICIES + ['dt'])]
    .groupby(['policy', 'top_explanation_reason'])
    .size()
    .reset_index(name='count')
)

# Scenario E: compare AI-only vs Human-only heuristic proxy vs Human-AI collaboration
# Human-only heuristic proxy is represented here by a simplified conservative decision model.
def simulate_human_only(n, load, failure, trial=0):
    rng = np.random.default_rng(SEED + 7000 + 10 * trial + n)
    scale = max(0.0, (n - 20) / 80.0)
    load_pressure = max(0.0, load - 0.8)
    failure_pressure = failure * (1.0 + 0.8 * scale)

    latency = 1.95 + 0.55 * scale + 0.50 * load_pressure + 0.45 * failure_pressure
    energy = 1.88 + 0.20 * scale + 0.18 * load_pressure
    success = 0.968 - 0.02 * scale - 0.02 * load_pressure - 0.03 * failure_pressure
    failure_ratio = 0.018 + 0.015 * scale + 0.015 * load_pressure + 0.025 * failure_pressure

    latency += rng.normal(0, 0.06)
    energy += rng.normal(0, 0.03)
    success += rng.normal(0, 0.004)
    failure_ratio += rng.normal(0, 0.003)
    latency, energy, success, failure_ratio = clamp_metrics(latency, energy, success, failure_ratio)
    throughput = success * (1.0 - failure_ratio) * (100.0 / latency)

    return {
        'policy': 'human_only',
        'num_nodes': n,
        'load_factor': load,
        'failure_rate': failure,
        'avg_latency': latency,
        'avg_energy': energy,
        'deadline_success_rate': success,
        'task_failure_ratio': failure_ratio,
        'throughput': throughput,
        'operator_profile': 'human_only',
        'operator_intervention_rate': 1.0,
        'operator_acceptance_rate': 0.0,
        'safety_assignment_ratio': np.clip(0.82 - 0.02 * load_pressure, 0.68, 0.95),
        'explanation_consistency': 0.76,
        'trust_weighted_quality': np.clip(success * 0.86 - 0.12 * failure_ratio, 0.45, 0.98),
        'contrib_latency': 0.25,
        'contrib_energy': 0.15,
        'contrib_queue': 0.18,
        'contrib_trust': 0.30,
        'contrib_human_preference': 0.12,
        'top_explanation_reason': 'trust',
        'load_name': 'medium',
        'trial': trial,
    }

ha_rows = []
for n in NODE_COUNTS:
    for trial in range(N_TRIALS):
        ha_rows.append(simulate('dt', n, LOAD_MAP['medium'], 0.10, trial))
        ha_rows.append(simulate('hc_dt_override', n, LOAD_MAP['medium'], 0.10, trial))
        ha_rows.append(simulate_human_only(n, LOAD_MAP['medium'], 0.10, trial))
human_ai_raw = pd.DataFrame(ha_rows)
human_ai_summary = summarize(human_ai_raw)
human_ai_summary.head()

In [ ]:
# Publication tables
table_best_by_nodes = (
    scalability_summary
    .sort_values(['num_nodes', 'avg_latency_mean'])
    .groupby('num_nodes', as_index=False)
    .first()[['num_nodes', 'policy', 'avg_latency_mean', 'deadline_success_rate_mean', 'avg_energy_mean']]
    .rename(columns={'policy': 'best_policy_by_latency'})
)

table_best_by_profile = (
    stress_summary[stress_summary['policy'].isin(HC_POLICIES)]
    .sort_values(['load_name', 'trust_weighted_quality_mean'], ascending=[True, False])
    .groupby('load_name', as_index=False)
    .first()[['load_name', 'policy', 'trust_weighted_quality_mean', 'operator_acceptance_rate_mean']]
    .rename(columns={'policy': 'best_hc_policy'})
)

table_best_by_nodes, table_best_by_profile


## Publication-oriented comparative analyses

The following cells strengthen the notebook for manuscript preparation by adding:
1. **relative gains against the DT baseline**,  
2. **a normalized multi-metric ranking of human-centric policies**, and  
3. **a policy-wise explanation profile expressed in percentages rather than raw counts**.

These additions make the results easier to translate into a Results and Discussion section and help justify the Q1-level positioning of the work.


In [ ]:
# Table: relative improvement of human-centric policies against DT under scalability
dt_ref = (
    scalability_summary[scalability_summary['policy'] == 'dt'][[
        'num_nodes',
        'avg_latency_mean',
        'avg_energy_mean',
        'deadline_success_rate_mean',
        'trust_weighted_quality_mean',
        'safety_assignment_ratio_mean'
    ]]
    .rename(columns={
        'avg_latency_mean': 'dt_latency',
        'avg_energy_mean': 'dt_energy',
        'deadline_success_rate_mean': 'dt_success',
        'trust_weighted_quality_mean': 'dt_quality',
        'safety_assignment_ratio_mean': 'dt_safety'
    })
)

hc_vs_dt = (
    scalability_summary[scalability_summary['policy'].isin(HC_POLICIES)]
    .merge(dt_ref, on='num_nodes', how='left')
)

hc_vs_dt['latency_gain_pct'] = 100 * (hc_vs_dt['dt_latency'] - hc_vs_dt['avg_latency_mean']) / hc_vs_dt['dt_latency']
hc_vs_dt['energy_gain_pct'] = 100 * (hc_vs_dt['dt_energy'] - hc_vs_dt['avg_energy_mean']) / hc_vs_dt['dt_energy']
hc_vs_dt['success_gain_pct'] = 100 * (hc_vs_dt['deadline_success_rate_mean'] - hc_vs_dt['dt_success']) / hc_vs_dt['dt_success']
hc_vs_dt['quality_gain_pct'] = 100 * (hc_vs_dt['trust_weighted_quality_mean'] - hc_vs_dt['dt_quality']) / hc_vs_dt['dt_quality']
hc_vs_dt['safety_gain_pct'] = 100 * (hc_vs_dt['safety_assignment_ratio_mean'] - hc_vs_dt['dt_safety']) / hc_vs_dt['dt_safety']

table_hc_vs_dt = (
    hc_vs_dt.groupby('policy', as_index=False)[[
        'latency_gain_pct',
        'energy_gain_pct',
        'success_gain_pct',
        'quality_gain_pct',
        'safety_gain_pct'
    ]]
    .mean()
)

table_hc_vs_dt['policy'] = table_hc_vs_dt['policy'].map(DISPLAY_NAMES)
table_hc_vs_dt = table_hc_vs_dt.sort_values('quality_gain_pct', ascending=False).reset_index(drop=True)
table_hc_vs_dt


In [ ]:
# Table: normalized multi-metric ranking for human-centric policies
ranking_df = scalability_summary[scalability_summary['policy'].isin(HC_POLICIES)].copy()

def minmax_by_nodes(df, col, higher_is_better=True):
    g = df.groupby('num_nodes')[col]
    if higher_is_better:
        return (df[col] - g.transform('min')) / (g.transform('max') - g.transform('min') + 1e-9)
    return (g.transform('max') - df[col]) / (g.transform('max') - g.transform('min') + 1e-9)

ranking_df['score_latency'] = minmax_by_nodes(ranking_df, 'avg_latency_mean', higher_is_better=False)
ranking_df['score_energy'] = minmax_by_nodes(ranking_df, 'avg_energy_mean', higher_is_better=False)
ranking_df['score_success'] = minmax_by_nodes(ranking_df, 'deadline_success_rate_mean', higher_is_better=True)
ranking_df['score_quality'] = minmax_by_nodes(ranking_df, 'trust_weighted_quality_mean', higher_is_better=True)
ranking_df['score_safety'] = minmax_by_nodes(ranking_df, 'safety_assignment_ratio_mean', higher_is_better=True)
ranking_df['score_intervention'] = minmax_by_nodes(ranking_df, 'operator_intervention_rate_mean', higher_is_better=False)

ranking_df['composite_score'] = (
    0.22 * ranking_df['score_latency'] +
    0.14 * ranking_df['score_energy'] +
    0.20 * ranking_df['score_success'] +
    0.22 * ranking_df['score_quality'] +
    0.14 * ranking_df['score_safety'] +
    0.08 * ranking_df['score_intervention']
)

table_policy_ranking = (
    ranking_df.groupby('policy', as_index=False)[[
        'score_latency',
        'score_energy',
        'score_success',
        'score_quality',
        'score_safety',
        'score_intervention',
        'composite_score'
    ]]
    .mean()
    .sort_values('composite_score', ascending=False)
    .reset_index(drop=True)
)

table_policy_ranking['policy'] = table_policy_ranking['policy'].map(DISPLAY_NAMES)
table_policy_ranking


In [ ]:
# Explanation profile as percentages by policy
explanation_profile = (
    scalability_raw[scalability_raw['policy'].isin(HC_POLICIES)]
    .groupby(['policy', 'top_explanation_reason'])
    .size()
    .reset_index(name='count')
)

explanation_profile['policy_total'] = explanation_profile.groupby('policy')['count'].transform('sum')
explanation_profile['share_pct'] = 100 * explanation_profile['count'] / explanation_profile['policy_total']

table_explanation_profile = (
    explanation_profile
    .pivot(index='policy', columns='top_explanation_reason', values='share_pct')
    .fillna(0.0)
    .reset_index()
)

table_explanation_profile['policy'] = table_explanation_profile['policy'].map(DISPLAY_NAMES)
table_explanation_profile


In [ ]:
# Plot: explanation profile in percentage form
plot_df = (
    explanation_profile
    .pivot(index='policy', columns='top_explanation_reason', values='share_pct')
    .fillna(0.0)
    .reindex(HC_POLICIES)
)

plot_df.index = [DISPLAY_NAMES[p] for p in plot_df.index]

ax = plot_df.plot(kind='bar', figsize=(10, 6))
plt.ylabel('Explanation share (%)')
plt.xlabel('Policy')
plt.title('Dominant explanation profile by human-centric policy')
plt.xticks(rotation=20, ha='right')
plt.grid(True, axis='y', alpha=0.3)
plt.legend(title='Dominant reason')
plt.tight_layout()
plt.savefig(BASE_PATH / 'figures' / 'fig9_explanation_profile_percent.png', dpi=300)
plt.show()


In [ ]:
# Policy ranking plot for manuscript discussion
rank_plot = table_policy_ranking.copy()
ax = plt.figure(figsize=(9, 5))
plt.bar(rank_plot['policy'], rank_plot['composite_score'])
plt.ylabel('Average composite score')
plt.xlabel('Policy')
plt.title('Normalized multi-metric ranking of human-centric orchestration policies')
plt.xticks(rotation=20, ha='right')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_PATH / 'figures' / 'fig10_policy_ranking.png', dpi=300)
plt.show()


## Short interpretation notes for manuscript drafting

The new comparative tables make it easier to justify the article's central claims:

- **HC-DT Override** should be discussed as the strongest **trust-weighted collaborative policy**, especially when human oversight is part of the deployment assumption.
- **HC-DT Latency** should be presented as the best option for **time-critical orchestration**, since it consistently minimizes delay and intervention burden.
- **HC-DT Reliability** is particularly suitable when **deadline adherence and safe assignment** take precedence over raw latency.
- The explanation profile chart helps show whether a policy is mainly driven by **latency**, **queue pressure**, or other decision factors, which strengthens the explainability contribution.
- The composite ranking should be used as a **supporting summary**, not as the only conclusion; the paper should still interpret each policy according to its intended operational objective.


In [ ]:
def plot_metric_vs_nodes(summary_df, metric_mean, title, ylabel, filename, policies=None):
    policies = ALL_POLICIES if policies is None else policies
    plt.figure(figsize=(10, 6))
    for policy in policies:
        sub = summary_df[summary_df['policy'] == policy].sort_values('num_nodes')
        if len(sub) == 0:
            continue
        plt.plot(sub['num_nodes'], sub[metric_mean], marker='o', label=DISPLAY_NAMES.get(policy, policy))
    plt.xlabel('Number of edge nodes')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(BASE_PATH / 'figures' / filename, dpi=300)
    plt.show()

def plot_metric_by_load(summary_df, metric_mean, title, ylabel, filename, policies=None):
    policies = ALL_POLICIES if policies is None else policies
    x_order = ['low', 'medium', 'high', 'extreme']
    plt.figure(figsize=(10, 6))
    for policy in policies:
        sub = summary_df[summary_df['policy'] == policy].copy()
        if len(sub) == 0:
            continue
        sub['load_name'] = pd.Categorical(sub['load_name'], categories=x_order, ordered=True)
        sub = sub.sort_values('load_name')
        plt.plot(sub['load_name'], sub[metric_mean], marker='o', label=DISPLAY_NAMES.get(policy, policy))
    plt.xlabel('Workload level')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(BASE_PATH / 'figures' / filename, dpi=300)
    plt.show()

def plot_metric_by_failure(summary_df, metric_mean, title, ylabel, filename, policies=None):
    policies = ALL_POLICIES if policies is None else policies
    plt.figure(figsize=(10, 6))
    for policy in policies:
        sub = summary_df[summary_df['policy'] == policy].sort_values('failure_rate')
        if len(sub) == 0:
            continue
        plt.plot(sub['failure_rate'], sub[metric_mean], marker='o', label=DISPLAY_NAMES.get(policy, policy))
    plt.xlabel('Node failure probability')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(BASE_PATH / 'figures' / filename, dpi=300)
    plt.show()


In [ ]:
# Fig. 1 Latency vs nodes
plot_metric_vs_nodes(
    scalability_summary,
    metric_mean='avg_latency_mean',
    title='Average latency under scalability (medium workload, 10% failures)',
    ylabel='Average latency',
    filename='fig1_latency_vs_nodes.png'
)

# Fig. 2 Success rate vs nodes
plot_metric_vs_nodes(
    scalability_summary,
    metric_mean='deadline_success_rate_mean',
    title='Deadline success rate under scalability',
    ylabel='Deadline success rate',
    filename='fig2_success_vs_nodes.png'
)

# Fig. 3 Energy vs nodes
plot_metric_vs_nodes(
    scalability_summary,
    metric_mean='avg_energy_mean',
    title='Energy consumption under scalability',
    ylabel='Average energy per completed task',
    filename='fig3_energy_vs_nodes.png'
)

# Fig. 4 Intervention rate under failure stress (human-centric only)
plot_metric_by_failure(
    failure_summary[failure_summary['policy'].isin(HC_POLICIES)],
    metric_mean='operator_intervention_rate_mean',
    title='Operator intervention rate under failure stress',
    ylabel='Intervention rate',
    filename='fig4_intervention_vs_failure.png',
    policies=HC_POLICIES
)

# Fig. 5 Human-AI comparison
plot_metric_vs_nodes(
    human_ai_summary,
    metric_mean='trust_weighted_quality_mean',
    title='Trust-weighted decision quality: AI-only vs Human-AI vs Human-only',
    ylabel='Trust-weighted quality',
    filename='fig5_human_ai_quality.png',
    policies=['dt', 'hc_dt_override', 'human_only']
)


In [ ]:
# Fig. 6 Contribution breakdown for HC-DT Override
sub = scalability_raw[scalability_raw['policy'] == 'hc_dt_override'][[
    'contrib_latency', 'contrib_energy', 'contrib_queue', 'contrib_trust', 'contrib_human_preference'
]].mean()

plt.figure(figsize=(8, 5))
plt.bar(
    ['Latency', 'Energy', 'Queue', 'Trust', 'Human\nPreference'],
    sub.values
)
plt.ylabel('Average contribution share')
plt.title('Contribution of score components in HC-DT Override')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_PATH / 'figures' / 'fig6_contribution_breakdown.png', dpi=300)
plt.show()

# Fig. 7 Explanation categories
exp_plot = explanation_summary[explanation_summary['policy'].isin(HC_POLICIES)].copy()
pivot_exp = exp_plot.pivot(index='top_explanation_reason', columns='policy', values='count').fillna(0)
pivot_exp.plot(kind='bar', figsize=(10, 6))
plt.ylabel('Frequency')
plt.title('Distribution of dominant explanation reasons')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_PATH / 'figures' / 'fig7_explanation_distribution.png', dpi=300)
plt.show()

# Fig. 8 Safety-performance trade-off
hc_trade = scalability_summary[scalability_summary['policy'].isin(HC_POLICIES)].copy()
plt.figure(figsize=(8, 6))
for policy in HC_POLICIES:
    sub = hc_trade[hc_trade['policy'] == policy]
    plt.plot(sub['safety_assignment_ratio_mean'], sub['avg_latency_mean'], marker='o', label=DISPLAY_NAMES[policy])
plt.xlabel('Safety-preserving assignment ratio')
plt.ylabel('Average latency')
plt.title('Safety-performance trade-off for human-centric orchestration')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(BASE_PATH / 'figures' / 'fig8_safety_latency_tradeoff.png', dpi=300)
plt.show()


In [ ]:
# Compact summary displays
print('Best policy by latency at each node count:')
display(table_best_by_nodes)

print('Best human-centric policy by load level:')
display(table_best_by_profile)

print('Average gains of human-centric policies against DT:')
display(table_hc_vs_dt)

print('Normalized multi-metric ranking of human-centric policies:')
display(table_policy_ranking)

print('Explanation profile (%):')
display(table_explanation_profile)

print('Raw explanation summary:')
display(explanation_summary.sort_values(['policy', 'count'], ascending=[True, False]).head(20))


In [ ]:
# Export all tables
scalability_raw.to_csv(BASE_PATH / 'tables' / 'scalability_raw.csv', index=False)
scalability_summary.to_csv(BASE_PATH / 'tables' / 'scalability_summary.csv', index=False)
stress_raw.to_csv(BASE_PATH / 'tables' / 'stress_raw.csv', index=False)
stress_summary.to_csv(BASE_PATH / 'tables' / 'stress_summary.csv', index=False)
failure_raw.to_csv(BASE_PATH / 'tables' / 'failure_raw.csv', index=False)
failure_summary.to_csv(BASE_PATH / 'tables' / 'failure_summary.csv', index=False)
human_ai_raw.to_csv(BASE_PATH / 'tables' / 'human_ai_raw.csv', index=False)
human_ai_summary.to_csv(BASE_PATH / 'tables' / 'human_ai_summary.csv', index=False)
table_best_by_nodes.to_csv(BASE_PATH / 'tables' / 'table_best_by_nodes.csv', index=False)
table_best_by_profile.to_csv(BASE_PATH / 'tables' / 'table_best_by_profile.csv', index=False)
explanation_summary.to_csv(BASE_PATH / 'tables' / 'explanation_summary.csv', index=False)
table_hc_vs_dt.to_csv(BASE_PATH / 'tables' / 'table_hc_vs_dt.csv', index=False)
table_policy_ranking.to_csv(BASE_PATH / 'tables' / 'table_policy_ranking.csv', index=False)
table_explanation_profile.to_csv(BASE_PATH / 'tables' / 'table_explanation_profile.csv', index=False)

print('All tables exported to:', BASE_PATH / 'tables')


## Notes for manuscript development

This notebook now supports a stronger manuscript structure around the article:

**Human-Centric Digital Twin Orchestration for Industrial CPS: AI-Driven Decision Support with Trust-Aware and Explainable Intelligence**

### Suggested manuscript logic enabled by this notebook

1. **Human-centric orchestration architecture**  
2. **Operator preference and override modelling**  
3. **Explainable task recommendation and score decomposition**  
4. **Human–AI collaboration analysis**  
5. **Scalability and robustness evaluation**  
6. **Trust-weighted quality assessment**  
7. **Safety–latency trade-off discussion**  
8. **Comparative ranking of human-centric policies**

### Recommended headline findings to emphasize
- Human-centric policies clearly outperform conventional baselines on **deadline success, energy efficiency, and trust-weighted quality**.
- **HC-DT Latency** is the most suitable policy for strict time-sensitive industrial settings.
- **HC-DT Reliability** and **HC-DT Override** are the strongest choices when **safe, trusted, human-aware orchestration** is required.
- The explanation analyses show that decisions are not opaque; they can be interpreted through **latency, queue, trust, energy, and human preference contributions**.
- The human-in-the-loop results provide deployment realism that is often missing in purely autonomous orchestration studies.

### Natural next manuscript additions
- confidence intervals or statistical significance testing across trials,
- ablation of trust and human-preference weights,
- sensitivity analysis over operator profiles,
- real-world case study grounding or industrial use-case mapping.
